# PINNs Framework — Demo: 2-D Viscous Burgers Equation

**PDE (scalar isotropic form):**  
$$\frac{\partial u}{\partial t} + u\,\frac{\partial u}{\partial x} + u\,\frac{\partial u}{\partial y} = \nu\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\right), \quad (x,y)\in[-1,1]^2,\; t\in[0,T]$$

**IC / BC:**  
$$u(0,x,y) = -\sin(\pi x)\sin(\pi y), \qquad u\big|_{\partial\Omega} = 0$$

The only structural difference from 1-D is:
- the network has **2 inputs** $(x, y)$ instead of 1
- two `PINSpatialDimension` objects are passed to the trainer
- `x` inside `residual_fn` is `(N, 2)` — column 0 is $x$, column 1 is $y$

In [ ]:
import sys
import os

sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import torch
import matplotlib.pyplot as plt

from pinns_framework import (
    PINArchitecture,
    PINSpatialDimension,
    PINLossFunction,
    PINTrainer,
    save_pinns,
    load_pinns,
)

torch.manual_seed(42)
print('torch:', torch.__version__)

## 1. Architecture

The network receives the spatial point $(x, y)$ concatenated into a 2-D
input vector, so `layers[0] = 2`.

In [ ]:
net = PINArchitecture(
    arch="MLP",
    params_arch={"layers": (2, 64, 64, 64, 1), "activation": "tanh"},
)
print(net)

## 2. Spatial Dimensions

Two `PINSpatialDimension` objects — one per axis.  The trainer samples
`n_colloc` points from each and **concatenates** them into an `(N, 2)`
collocation tensor that is passed (with `requires_grad=True`) to the
residual function.

In [ ]:
x_dim = PINSpatialDimension(name="x", domain=(-1.0, 1.0), n_colloc=2500)
y_dim = PINSpatialDimension(name="y", domain=(-1.0, 1.0), n_colloc=2500)
print(x_dim)
print(y_dim)

## 3. PDE Residual Function

Inside `residual_fn`, `x` has shape `(N, 2)`:
- `x[:, 0:1]` — x-coordinate
- `x[:, 1:2]` — y-coordinate

A single `autograd.grad` call on the concatenated tensor gives both
first-order partial derivatives at once:

```python
grad_u = torch.autograd.grad(u.sum(), x, create_graph=True)[0]  # (N, 2)
du_dx  = grad_u[:, 0:1]   # ∂u/∂x
du_dy  = grad_u[:, 1:2]   # ∂u/∂y
```

Second derivatives are extracted the same way from the next `grad` call.

In [ ]:
def burgers2d_residual_td(u, u_prev, x, dt, nu):
    """
    2-D scalar Burgers residual for Backward Euler time discretisation.
    x is (N, 2): column 0 = x-coord, column 1 = y-coord.
    """
    # First-order partials — one grad call gives both ∂/∂x and ∂/∂y
    grad_u = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    du_dx  = grad_u[:, 0:1]
    du_dy  = grad_u[:, 1:2]

    # Second-order partials via a second grad call on each column
    du_dxx = torch.autograd.grad(du_dx.sum(), x, create_graph=True)[0][:, 0:1]
    du_dyy = torch.autograd.grad(du_dy.sum(), x, create_graph=True)[0][:, 1:2]

    return (u - u_prev) / dt + u * du_dx + u * du_dy - nu * (du_dxx + du_dyy)

## 4. Loss Function & Trainer

In [ ]:
loss_fn = PINLossFunction(
    spatial_dims=[x_dim, y_dim],
    residual_fn=burgers2d_residual_td,
    form="strong",
)

trainer = PINTrainer(
    net=net,
    scheme="TD-PINNS",
    loss_fn=loss_fn,
    spatial_dims=[x_dim, y_dim],
    epochs=5000,
    dt=0.05,
    time_integrator="BE",
    lr=1e-3,
    tolerance=1e-4,
    ic_weight=10.0,
    bc_weight=10.0,
    pde_weight=1.0,
    verbose=True,
    log_every=1000,
)

print(loss_fn)
print(f'device: {trainer.device}')

## 5. Initial and Boundary Conditions

**IC** is sampled on a regular grid so it covers the full domain uniformly.  
**BC** is sampled on all four edges of $[-1,1]^2$.

Both are passed to `trainer.fit` as `(N, 2)` coordinate tensors.

In [ ]:
# ── Initial condition: u(0, x, y) = -sin(πx)sin(πy) ─────────────────────
n_ic = 40                                                  # 40×40 = 1600 pts
xy   = torch.linspace(-1.0, 1.0, n_ic)
xx, yy = torch.meshgrid(xy, xy, indexing='ij')             # (n_ic, n_ic)

x_ic = torch.stack([xx.flatten(), yy.flatten()], dim=1)    # (1600, 2)
u_ic = (-torch.sin(np.pi * xx) * torch.sin(np.pi * yy)).flatten().unsqueeze(1)

# ── Boundary condition: u = 0 on all four edges ───────────────────────────
n_bc  = 80
edge  = torch.linspace(-1.0, 1.0, n_bc)
ones  = torch.ones(n_bc)

x_edges = torch.cat([edge,  edge, -ones,  ones])
y_edges = torch.cat([-ones,  ones,  edge,  edge])
x_bc    = torch.stack([x_edges, y_edges], dim=1)           # (4*n_bc, 2)
u_bc    = torch.zeros(4 * n_bc, 1)

# ── Plot IC ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
u_ic_grid = u_ic.reshape(n_ic, n_ic).numpy()
im = ax.pcolormesh(xy.numpy(), xy.numpy(), u_ic_grid, cmap='RdBu_r', shading='auto')
plt.colorbar(im, ax=ax, label='u')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Initial condition  $u(0,x,y) = -\\sin(\\pi x)\\sin(\\pi y)$')
plt.tight_layout(); plt.show()

## 6. Train

`trainer.fit` always returns `list[nn.Module]` — one network per time level,
index 0 being the IC network.

In [ ]:
T  = 0.5
nu = 0.01 / np.pi

trained_nets = trainer.fit(
    x_ic=x_ic,
    u_ic=u_ic,
    x_bc=x_bc,
    u_bc=u_bc,
    T=T,
    nu=nu,
)

## 7. Evaluate & Plot

In [ ]:
dev = trainer.device

# Evaluation grid
n_eval = 60
xy_eval = torch.linspace(-1.0, 1.0, n_eval)
xx_e, yy_e = torch.meshgrid(xy_eval, xy_eval, indexing='ij')
x_eval = torch.stack([xx_e.flatten(), yy_e.flatten()], dim=1).to(dev)

# Pick a subset of time steps to display
n_nets = len(trained_nets)
dt     = trainer.dt
show_steps = [0, n_nets // 4, n_nets // 2, 3 * n_nets // 4, n_nets - 1]
show_steps = sorted(set(show_steps))  # deduplicate

fig, axes = plt.subplots(1, len(show_steps), figsize=(4 * len(show_steps), 3.5))

vmax = u_ic.abs().max().item()

for ax, step in zip(axes, show_steps):
    trained_nets[step].eval()
    with torch.no_grad():
        u_pred = trained_nets[step](x_eval).cpu().reshape(n_eval, n_eval).numpy()
    im = ax.pcolormesh(
        xy_eval.numpy(), xy_eval.numpy(), u_pred,
        cmap='RdBu_r', shading='auto', vmin=-vmax, vmax=vmax,
    )
    ax.set_title(f't = {step * dt:.2f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)

fig.suptitle('TD-PINNs — 2-D Burgers equation', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Diagonal slice x = y for a space-time view
diag  = torch.linspace(-1.0, 1.0, 200)
xy_diag = torch.stack([diag, diag], dim=1).to(dev)

U_diag = []
for net_i in trained_nets:
    net_i.eval()
    with torch.no_grad():
        U_diag.append(net_i(xy_diag).cpu().numpy().squeeze())

U_diag = np.array(U_diag)          # (n_steps+1, 200)
t_ax   = np.linspace(0, T, n_nets)

fig, ax = plt.subplots(figsize=(7, 4))
pcm = ax.pcolormesh(diag.numpy(), t_ax, U_diag, cmap='RdBu_r', shading='auto')
plt.colorbar(pcm, ax=ax, label='u')
ax.set_xlabel('s  (along diagonal x = y)')
ax.set_ylabel('t')
ax.set_title('Space–time slice along x = y')
plt.tight_layout(); plt.show()

## 8. Saving & Loading

`load_pinns` is scheme- and architecture-agnostic — always returns `list[nn.Module]`.

In [ ]:
SAVE_PATH   = "burgers2d_tdpinns_mlp.pt"
ARCH        = "MLP"
PARAMS_ARCH = {"layers": (2, 64, 64, 64, 1), "activation": "tanh"}

save_pinns(trained_nets, trainer, ARCH, PARAMS_ARCH, SAVE_PATH)

loaded_nets = load_pinns(SAVE_PATH)

# Round-trip check on a few IC points
x_check = x_ic[:5].to(dev)
with torch.no_grad():
    orig     = trained_nets[0](x_check).cpu().numpy()
    reloaded = loaded_nets[0](x_check).numpy()

print('Round-trip outputs match:', np.allclose(orig, reloaded))